# 92. Reverse Linked List II

**Difficulty**: Medium  
**Topics**: Linked List  
**Link**: [LeetCode 92](https://leetcode.com/problems/reverse-linked-list-ii/)

---

## Problem Statement

Given the `head` of a singly linked list and two integers `left` and `right` where `left <= right`, reverse the nodes of the list from position `left` to position `right`, and return the reversed list.

### Examples

**Example 1:**
```
Input: head = [1,2,3,4,5], left = 2, right = 4
Output: [1,4,3,2,5]
```

**Example 2:**
```
Input: head = [5], left = 1, right = 1
Output: [5]
```

### Constraints

- The number of nodes in the list is `n`.
- `1 <= n <= 500`
- `1 <= left <= right <= n`

In [ ]:
class ListNode:
    def __init__(self, val=0, next=None):
        self.val = val
        self.next = next
    
    def __repr__(self):
        result = []
        node = self
        while node:
            result.append(str(node.val))
            node = node.next
        return " -> ".join(result)

def create_list(values):
    """Helper to create linked list from array"""
    if not values:
        return None
    head = ListNode(values[0])
    current = head
    for val in values[1:]:
        current.next = ListNode(val)
        current = current.next
    return head

### Algorithm Visualization

```
Input: [1,2,3,4,5], left=2, right=4

Initial setup:
dummy -> 1 -> 2 -> 3 -> 4 -> 5
         ^    ^    ^    ^    ^
        prev  curr  next

Step 1: Move prev to position left-1 (node 1)
dummy -> 1 -> 2 -> 3 -> 4 -> 5
         ^    ^    ^    ^    ^
        prev  curr  next

Step 2: Extract and insert (right-left = 2 times)

Iteration 1: Extract node 3
- curr refers to node 2 (same object), but its position shifts right
- next = curr.next = 3
- Extract: 2.next = 3.next (2 -> 4)
- Insert: 3.next = prev.next (3 -> 2), prev.next = 3 (1 -> 3)
Result: dummy -> 1 -> 3 -> 2 -> 4 -> 5
         ^    ^    ^    ^    ^
        prev  curr  next
Note: curr (node 2) moved from position 2 to position 3 relative to prev

Iteration 2: Extract node 4
- curr still refers to node 2, position shifts further right
- next = curr.next = 4
- Extract: 2.next = 4.next (2 -> 5)
- Insert: 4.next = prev.next (4 -> 3), prev.next = 4 (1 -> 4)
Result: dummy -> 1 -> 4 -> 3 -> 2 -> 5
         ^    ^    ^    ^    ^
        prev  curr  next
Note: curr (node 2) moved from position 3 to position 4 relative to prev

Key insight: 
- curr always points to the same node (originally node 2)
- But curr's relative position moves right as we extract nodes before it
- prev always points to the same node (originally node 1)
```

In [ ]:
def reverse_between_extraction(head: ListNode, left: int, right: int) -> ListNode:
    """Node Extraction Method: O(n) time, O(1) space"""
    dummy = ListNode(0, head)
    prev = dummy
    
    # Move prev to position left-1
    for _ in range(left - 1):
        prev = prev.next
    
    # curr is the first node of the sublist to reverse
    curr = prev.next
    
    # Extract and insert nodes (right-left times)
    for _ in range(right - left):
        # Extract next node
        next_node = curr.next
        
        # Remove it from current position
        curr.next = next_node.next
        
        # Insert it after prev
        next_node.next = prev.next
        prev.next = next_node
    
    return dummy.next

# Test
head = create_list([1, 2, 3, 4, 5])
print(f"Original: {head}")
reversed_head = reverse_between_extraction(head, 2, 4)
print(f"Reversed [2,4]: {reversed_head}")  # [1,4,3,2,5]

# Test edge case: single node
head2 = create_list([5])
print(f"\nOriginal: {head2}")
reversed_head2 = reverse_between_extraction(head2, 1, 1)
print(f"Reversed [1,1]: {reversed_head2}")  # [5]

---

## Approach 2: Three Pointer Method (Traditional Reversal)

### Intuition

Use three pointers (prev, curr, next) to reverse the sublist in-place, similar to how we reverse an entire list.

### Why it works

The three-pointer method reverses the direction of links within the sublist. We need to carefully reconnect the boundaries.

### Algorithm Visualization

```
Input: [1,2,3,4,5], left=2, right=4

Step 1: Find boundaries
prev_sublist -> 2 -> 3 -> 4 -> 5
              ^    ^    ^    ^
              prev curr next

Step 2: Reverse sublist with three pointers
prev_sublist -> 2 <- 3 <- 4    5
              ^    ^    ^    ^
              prev curr next

Step 3: Reconnect boundaries
1 -> 4 -> 3 -> 2 -> 5
```

### Complexity
- **Time**: O(n)
- **Space**: O(1)

In [ ]:
def reverse_between_three_pointers(head: ListNode, left: int, right: int) -> ListNode:
    """Three Pointer Method: O(n) time, O(1) space"""
    dummy = ListNode(0, head)
    
    # Find node before sublist
    prev_sublist = dummy
    for _ in range(left - 1):
        prev_sublist = prev_sublist.next
    
    # Reverse sublist using three pointers
    prev = None
    curr = prev_sublist.next
    
    for _ in range(right - left + 1):
        next_node = curr.next
        curr.next = prev
        prev = curr
        curr = next_node
    
    # Reconnect boundaries
    prev_sublist.next.next = curr  # Original start connects to after sublist
    prev_sublist.next = prev      # prev_sublist connects to new head
    
    return dummy.next

# Test
head = create_list([1, 2, 3, 4, 5])
print(f"Original: {head}")
reversed_head = reverse_between_three_pointers(head, 2, 4)
print(f"Reversed [2,4]: {reversed_head}")  # [1,4,3,2,5]

def reverse_between_recursive(head: ListNode, left: int, right: int) -> ListNode:
    """Recursive: O(n) time, O(n) space"""
    if not head or left == right:
        return head
    
    dummy = ListNode(0, head)
    left_node = dummy
    stop = False
    
    def recurse_and_reverse(node, position):
        nonlocal left_node, stop
        if position == left:
            left_node = node
        
        if position == right:
            # Swap values
            left_node.val, node.val = node.val, left_node.val
            left_node = left_node.next
            stop = True
            return
        
        if not stop and node.next:
            recurse_and_reverse(node.next, position + 1)
            
            if not stop and position >= left:
                # Swap values
                left_node.val, node.val = node.val, left_node.val
                left_node = left_node.next
    
    recurse_and_reverse(head, 1)
    return dummy.next

In [ ]:
def reverse_between_recursive(head: ListNode, left: int, right: int) -> ListNode:
    """Recursive: O(n) time, O(n) space"""
    if not head or left == right:
        return head
    
    dummy = ListNode(0, head)
    self.left_node = dummy
    self.stop = False
    
    def recurse_and_reverse(node, position):
        if position == left:
            self.left_node = node
        
        if position == right:
            # Swap values
            self.left_node.val, node.val = node.val, self.left_node.val
            self.left_node = self.left_node.next
            self.stop = True
            return
        
        if not self.stop and node.next:
            recurse_and_reverse(node.next, position + 1)
            
            if not self.stop and position >= left:
                # Swap values
                self.left_node.val, node.val = node.val, self.left_node.val
                self.left_node = self.left_node.next
    
    recurse_and_reverse(head, 1)
    return dummy.next

# Test
head = create_list([1, 2, 3, 4, 5])
print(f"Original: {head}")
reversed_head = reverse_between_recursive(head, 2, 4)
print(f"Reversed [2,4]: {reversed_head}")

---

## Approach 4: Stack

### Intuition

1. First pass: find nodes from left to right and push to stack
2. Second pass: pop from stack and update values

### Complexity
- **Time**: O(n)
- **Space**: O(right-left) - stack stores sublist

In [ ]:
def reverse_between_stack(head: ListNode, left: int, right: int) -> ListNode:
    """Stack: O(n) time, O(right-left) space"""
    stack = []
    curr = head
    position = 1
    
    # First pass: collect nodes to reverse
    while curr and position <= right:
        if position >= left:
            stack.append(curr.val)
        curr = curr.next
        position += 1
    
    # Second pass: update values in reverse order
    curr = head
    position = 1
    while curr and position <= right:
        if position >= left:
            curr.val = stack.pop()
        curr = curr.next
        position += 1
    
    return head

# Test
head = create_list([1, 2, 3, 4, 5])
print(f"Original: {head}")
reversed_head = reverse_between_stack(head, 2, 4)
print(f"Reversed [2,4]: {reversed_head}")

---

## Detailed Comparison: Extraction vs Three-Pointer

### Node Extraction Method (Link Hopping)
```
Key Idea: Extract next node and insert at front

dummy -> 1 -> 2 -> 3 -> 4 -> 5
         ^    ^    ^
        prev  curr  next

Extract 3: 1 -> 3 -> 2 -> 4 -> 5
Extract 4: 1 -> 4 -> 3 -> 2 -> 5

Pros:
- Intuitive "building reversed list" concept
- No need to track three pointers during reversal
- Easier to visualize

Cons:
- Slightly more complex insertion logic
```

### Three-Pointer Method (Traditional Reversal)
```
Key Idea: Reverse links in-place, then reconnect

1 -> 2 -> 3 -> 4 -> 5
    ^    ^    ^
   prev curr next

During reversal:
1 -> 2 <- 3 <- 4    5
         ^    ^    ^
        prev curr next

After reconnection:
1 -> 4 -> 3 -> 2 -> 5

Pros:
- Same pattern as full list reversal
- Clear separation of reversal and reconnection
- More familiar to those who know LeetCode 206

Cons:
- Need to carefully manage boundary connections
- More pointer manipulation
```

---

## Walkthrough: Node Extraction Method

Input: `[1,2,3,4,5]`, `left=2`, `right=4`

**Step 1: Initialize**
- dummy -> 1 -> 2 -> 3 -> 4 -> 5
- Move prev to position 1 (left-1)
- curr = prev.next = node 2

**Step 2: Extract and Insert (2 iterations)**

**Iteration 1:**
- Extract: next_node = curr.next = node 3
- Remove: curr.next = next_node.next (2 -> 4)
- Insert: next_node.next = prev.next (3 -> 2), prev.next = next_node (1 -> 3)
- Result: 1 -> 3 -> 2 -> 4 -> 5

**Iteration 2:**
- Extract: next_node = curr.next = node 4
- Remove: curr.next = next_node.next (2 -> 5)
- Insert: next_node.next = prev.next (4 -> 3), prev.next = next_node (1 -> 4)
- Result: 1 -> 4 -> 3 -> 2 -> 5

In [ ]:
def reverse_between_verbose_extraction(head: ListNode, left: int, right: int) -> ListNode:
    """Verbose Node Extraction Method"""
    dummy = ListNode(0, head)
    prev = dummy
    
    print(f"Initial: {dummy.next}")
    print(f"Reversing from position {left} to {right}")
    
    # Move prev to position left-1
    for i in range(left - 1):
        prev = prev.next
        print(f"Step {i+1}: prev moved to {prev.val}")
    
    curr = prev.next
    print(f"Starting extraction at curr={curr.val}")
    
    # Extract and insert
    for i in range(right - left):
        print(f"\nIteration {i+1}:")
        print(f"  Before: {dummy.next}")
        
        next_node = curr.next
        curr.next = next_node.next
        next_node.next = prev.next
        prev.next = next_node
        
        print(f"  After:  {dummy.next}")
        print(f"  Extracted node {next_node.val} and inserted after {prev.val}")
    
    print(f"\nFinal result: {dummy.next}")
    return dummy.next

head = create_list([1, 2, 3, 4, 5])
reverse_between_verbose_extraction(head, 2, 4)

---

## Comparison

| Approach | Time | Space | Pros | Cons |
|----------|------|-------|------|------|
| **Node Extraction** | O(n) | O(1) | Intuitive, building concept | Complex insertion logic |
| **Three Pointer** | O(n) | O(1) | Familiar pattern, clear phases | Boundary management |
| **Recursive** | O(n) | O(n) | Elegant logic | Stack overflow risk |
| **Stack** | O(n) | O(right-left) | Simple to understand | Extra space usage |

**Best choice**: **Node Extraction** for intuition, **Three Pointer** for familiarity.

---

## Edge Cases

| Case | Input | Output | Reason |
|------|-------|--------|--------|
| **Single node** | [5], 1, 1 | [5] | No reversal needed |
| **Reverse all** | [1,2,3], 1, 3 | [3,2,1] | Entire list reversed |
| **Reverse first two** | [1,2,3,4], 1, 2 | [2,1,3,4] | Only first pair |
| **Reverse last two** | [1,2,3,4], 3, 4 | [1,2,4,3] | Only last pair |
| **Middle only** | [1,2,3,4,5], 2, 4 | [1,4,3,2,5] | Middle section |
| **No reversal** | [1,2,3], 2, 2 | [1,2,3] | left == right |

---

## Common Mistakes

1. **Off-by-one errors in positioning**
   ```python
   # WRONG: Moves to left instead of left-1
   for _ in range(left):
       prev = prev.next
   # CORRECT: Move to position left-1
   for _ in range(left - 1):
       prev = prev.next
   ```

2. **Incorrect loop count for reversal**
   ```python
   # WRONG: Reverses one extra node
   for _ in range(right - left + 1):
   # CORRECT: right-left iterations
   for _ in range(right - left):
   ```

3. **Losing node connections**
   ```python
   # WRONG: Overwrites before saving
   curr.next = next_node.next  # Lost connection!
   # CORRECT: Save next_node first
   next_node = curr.next
   ```

---

## Related Problems

| Problem | Difficulty | Pattern |
|---------|------------|--------|
| [206. Reverse Linked List](https://leetcode.com/problems/reverse-linked-list/) | Easy | Full list reversal |
| [25. Reverse Nodes in k-Group](https://leetcode.com/problems/reverse-nodes-in-k-group/) | Hard | Generalization |
| [24. Swap Nodes in Pairs](https://leetcode.com/problems/swap-nodes-in-pairs/) | Medium | Fixed size reversal |
| [61. Rotate List](https://leetcode.com/problems/rotate-list/) | Medium | List manipulation |

---

## Key Takeaways

| Pattern | When to Use |
|---------|-------------|
| **Node Extraction** | Building reversed list concept |
| **Three Pointer** | Traditional in-place reversal |
| **Dummy head** | When head might change | 
| **Move to left-1** | Always position before sublist |
| **right-left iterations** | Number of reversals needed |
| **Preserve connections** | Keep boundaries intact |

**Remember**: Node extraction builds the reversed list piece by piece, while three-pointer reverses in-place then reconnects!